In [3]:
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from data_prep.RAGEvaluator import evaluate
from data_prep.items import Item
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

In [4]:
load_dotenv(override=True)
DB = "products_vectorstore"

In [5]:
hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [6]:
username = "leearum95"
dataset = f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 4,000 validation items, 3,099 test items


In [7]:
for item in train[0:10]:
    print(item.category)

Luggage & Bags
Arts & Entertainment
Cameras & Optics
Office Supplies
Religious & Ceremonial
Sporting Goods
Toys & Games
Arts & Entertainment
Arts & Entertainment
Apparel & Accessories


In [8]:
client = chromadb.PersistentClient(path=DB)


In [9]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [10]:
# Check if the collection exists; if not, create it

collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 1000)):
        documents = [item.summary for item in train[i: i+1000]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category} for item in train[i: i+1000]]
        ids = [f"doc_{j}" for j in range(i, i+1000)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

In [11]:

import matplotlib.colors as mcolors
import matplotlib.cm as cm

MAXIMUM_DATAPOINTS = 10_000

CATEGORIES = ['Bundles', 'Food, Beverages & Tobacco', 'Product Add-Ons', 'Gift Cards', 'Hardware', 'Home & Garden', 'Sporting Goods', 'Electronics', 'Baby & Toddler', 'Uncategorized', 'Apparel & Accessories', 'Furniture', 'Media', 'Toys & Games', 'Religious & Ceremonial', 'Luggage & Bags', 'Cameras & Optics', 'Arts & Entertainment', 'Software', 'Office Supplies', 'Animals & Pet Supplies', 'Vehicles & Parts', 'Health & Beauty', 'Business & Industrial', 'Services']


# Generate unique colors for each category using multiple colormaps for better distinction
def generate_distinct_colors(n_colors):
    """Generate n distinct colors using multiple colormaps"""
    colors = []
    
    # Use tab20 for first 20 colors
    if n_colors <= 20:
        cmap = cm.get_cmap('tab20')
        colors = [mcolors.to_hex(cmap(i / 20)) for i in range(n_colors)]
    else:
        # For more colors, combine multiple colormaps
        cmap1 = cm.get_cmap('tab20')  # 20 colors
        cmap2 = cm.get_cmap('Set1')   # 9 colors  
        cmap3 = cm.get_cmap('Set2')   # 8 colors
        
        # First 20 from tab20
        colors.extend([mcolors.to_hex(cmap1(i / 20)) for i in range(20)])
        
        # Next 9 from Set1
        if n_colors > 20:
            remaining = min(n_colors - 20, 9)
            colors.extend([mcolors.to_hex(cmap2(i / 9)) for i in range(remaining)])
            
        # Next 8 from Set2 if still need more
        if n_colors > 29:
            remaining = n_colors - 29
            colors.extend([mcolors.to_hex(cmap3(i / 8)) for i in range(min(remaining, 8))])
    
    return colors[:n_colors]

COLORS = generate_distinct_colors(len(CATEGORIES))

result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

/var/folders/_t/4vpfft894ddd5ymy303fdf2m0000gn/T/ipykernel_6965/3436375814.py:20: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap1 = cm.get_cmap('tab20')  # 20 colors
/var/folders/_t/4vpfft894ddd5ymy303fdf2m0000gn/T/ipykernel_6965/3436375814.py:21: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap2 = cm.get_cmap('Set1')   # 9 colors
/var/folders/_t/4vpfft894ddd5ymy303fdf2m0000gn/T/ipykernel_6965/3436375814.py:22: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
 

In [12]:

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [13]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [14]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [15]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [16]:
def vector(item):
    return encoder.encode(item.summary)

In [17]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['category'] for m in results['metadatas'][0][:]]
    return documents, prices

In [18]:
find_similars(test[0])

(['The RME HDSPe AIO Pro is a professional 30-channel PCIe audio interface featuring simultaneous analog, digital, MIDI, and high-resolution I/O capabilities, advanced clock technology, flexible routing, and enhanced output levels, ideal for studio and broadcast applications.',
  'The iConnectivity mioXM MIDI Interface Patchbay seamlessly integrates and manages multiple MIDI connections, enabling versatile control and connectivity for music production and live performance setups.',
  'The Korg Microkey Air 49 is a wireless MIDI controller with a compact natural touch mini keyboard, Bluetooth Low Energy connectivity for iOS, Mac, and Windows devices, and includes comprehensive software bundles for versatile music production and performance.',
  'A versatile 5.1 channel optical USB sound controller supporting stereo PCM, digital and analog recording, copy protection, multiple input/output options, and seamless plug-and-play compatibility across Windows and Mac systems for immersive audio

In [19]:
def make_context(similars, categories):
    message = "For context, here are some other items that might be similar to the item you need to classify.\n\n"
    for similar, category in zip(similars, categories):
        message += f"Potentially related product:\n{similar}\nCategory is {category}\n\n"
    return message

In [20]:
documents, categories = find_similars(test[0])
print(make_context(documents, categories))

For context, here are some other items that might be similar to the item you need to classify.

Potentially related product:
The RME HDSPe AIO Pro is a professional 30-channel PCIe audio interface featuring simultaneous analog, digital, MIDI, and high-resolution I/O capabilities, advanced clock technology, flexible routing, and enhanced output levels, ideal for studio and broadcast applications.
Category is Electronics

Potentially related product:
The iConnectivity mioXM MIDI Interface Patchbay seamlessly integrates and manages multiple MIDI connections, enabling versatile control and connectivity for music production and live performance setups.
Category is Arts & Entertainment

Potentially related product:
The Korg Microkey Air 49 is a wireless MIDI controller with a compact natural touch mini keyboard, Bluetooth Low Energy connectivity for iOS, Mac, and Windows devices, and includes comprehensive software bundles for versatile music production and performance.
Category is Arts & En

In [21]:
def messages_for(item, similars, categories):
    message = f"Classfify the following product. Respond with the category, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, categories)
    return [{"role": "user", "content": message}]

In [22]:
documents, categories = find_similars(test[0])
print(messages_for(test[0], documents, categories)[0]['content'])

Classfify the following product. Respond with the category, no explanation

The Akai Professional APC64 is an advanced Ableton Live MIDI controller with 64 RGB velocity-sensitive pads, touch strips, internal step sequencer, real-time visual feedback, and comprehensive hardware control, designed for music production, live performance, and creative experimentation.

For context, here are some other items that might be similar to the item you need to classify.

Potentially related product:
The RME HDSPe AIO Pro is a professional 30-channel PCIe audio interface featuring simultaneous analog, digital, MIDI, and high-resolution I/O capabilities, advanced clock technology, flexible routing, and enhanced output levels, ideal for studio and broadcast applications.
Category is Electronics

Potentially related product:
The iConnectivity mioXM MIDI Interface Patchbay seamlessly integrates and manages multiple MIDI connections, enabling versatile control and connectivity for music production and li

In [ ]:

def gpt_5__1_rag(item):
    documents, categories = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, categories), reasoning_effort="none", seed=42)
    return response.choices[0].message.content

In [24]:
print(gpt_5__1_rag(test[0]))
print(test[0].category)

Arts & Entertainment
Arts & Entertainment


In [25]:
evaluate(gpt_5__1_rag,test, size =200, workers=1)

  0%|          | 0/200 [00:00<?, ?it/s]

TypeError: 'Item' object is not subscriptable